# Envelopes: hoe een geluid begint en eindigt

In les 1 heb je een sinusgolf laten uitsterven met `np.exp(-decay * t)`. In les 2 hebben we datzelfde trucje hergebruikt om een melodie wat "aangeslagen" te laten klinken in plaats van als een orgel.

Maar een trommel klinkt anders dan een gitaar, en een gitaar klinkt anders dan een viool. Niet alleen door de boventonen (les 2), ook door **hoe het geluid begint en eindigt in de tijd**. Een trommel is direct luid en meteen weg. Een viool zwelt aan. Een gitaar slaat hard aan en sterft langzaam uit.

Die "vorm in de tijd" heet een **envelope**. In deze les gaan we er een paar bouwen, eerst stap voor stap met een lus, daarna met numpy. 

## 1. Een envelope is gewoon een array

Een envelope is een array van amplitudes — voor elke sample één waarde tussen 0 en 1. Je vermenigvuldigt die elementsgewijs met je golf, en klaar.

Klinkt eenvoudig? Dat is het ook. We hebben het in les 1 al gedaan, maar zonder de naam.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd

SAMPLERATE = 16000
duration = 1.0
t = np.linspace(0, duration, int(SAMPLERATE * duration), endpoint=False)

Wat denk je dat `len(t)` teruggeeft? En waarom?

In [ ]:
print(len(t))

De waarde is 16000. Dat zijn `SAMPLERATE * duration` samples — één seconde aan 16000 samples per seconde.

Wat verwacht je dat de waarde van de eerste sample `t[0]` is? En van de laatste `t[-1]`? 

In [ ]:
print(t[0], t[-1])

`t[0]` is 0.0 en `t[-1]` is iets net onder 1.0. Met `endpoint=False` slaan we de eindwaarde over — dat is precies wat we willen, anders zou sample 0 én sample 16000 allebei tijdstip 1.0 zijn (één te veel).

Nu we het tijdsverloop hebben, kunnen we een sinusgolf maken.

In [ ]:
wave = np.sin(2 * np.pi * 50 * t)
envelope = 1.0 - t / duration  # van 1 naar 0, lineair

plt.figure(figsize=(10, 4))
plt.plot(t, wave, alpha=0.4, label="golf")
plt.legend()
plt.xlabel("Tijd (s)")
plt.show()

Wat we vandaag doen, is die golf een envelope geven. We gaan de amplitude niet statisch veranderen, maar over het verloop van de tijd. Een eenvoudige manier om dat te doen is lineair.

In [ ]:
wave = np.sin(2 * np.pi * 50 * t)
envelope = 1.0 - t / duration  # van 1 naar 0, lineair

plt.figure(figsize=(10, 4))
plt.plot(t, wave, alpha=0.4, label="golf")
plt.plot(t, envelope, color="red", linewidth=2, label="envelope")
plt.plot(t, wave * envelope, color="black", label="product")
plt.legend()
plt.xlabel("Tijd (s)")
plt.show()

De rode lijn is de envelope: de "buitenste vorm" waarbinnen de golf moet blijven. Het zwarte signaal is `wave * envelope`: een sinus die geleidelijk uitsterft.

In de regel `wave * envelope` — wat doet de `*` hier eigenlijk? Het zijn twee numpy-arrays. Wat zou er gebeuren als ze niet even lang zouden zijn?

In [ ]:
# probeer eens wat er gebeurt:
np.array([1, 2, 3]) * np.array([10, 20])

Numpy klaagt: de vormen passen niet bij elkaar. Bij vector-vector vermenigvuldiging *moet* de lengte gelijk zijn. Onthoud dat — dit is een veelvoorkomende fout als je je envelopes en golven mixt.

## 2. Een envelope opbouwen met een lus

Voor we ingewikkelde envelopes maken, beginnen we met de saaiste van allemaal: een envelope die overal 1.0 is. Geen effect, dus. Maar het is een goede manier om de structuur onder de knie te krijgen.

In [ ]:
def make_constant_envelope(duration, level=1.0):
    n_samples = int(SAMPLERATE * duration)
    envelope = np.zeros(n_samples)
    for n in range(n_samples):
        envelope[n] = level
    return envelope

env = make_constant_envelope(1.0)
print(env.shape, env[0], env[-1])

Drie dingen om bij stil te staan voor we verder gaan.

### 1

In `for n in range(n_samples):` — wat is de waarde van `n` bij de *eerste* iteratie? En bij de *laatste*? Hoeveel iteraties zijn er in totaal?

In [ ]:
# controleer:
for n in range(5):
    print(n)

Eerste iteratie: `n = 0`. Laatste: `n = n_samples - 1`. Totaal: `n_samples` iteraties. **Niet** `n_samples + 1`, en **niet** beginnend bij 1.

### 2

Wat gebeurt er als je `range(n_samples + 1)` zou schrijven in plaats van `range(n_samples)`? Probeer het.

In [ ]:
# pas make_constant_envelope expres fout aan en kijk wat er gebeurt
def make_broken_envelope(duration, level=1.0):
    n_samples = int(SAMPLERATE * duration)
    envelope = np.zeros(n_samples)
    for n in range(n_samples + 1):
        envelope[n] = level
    return envelope

make_broken_envelope(1.0)

Je krijgt een `IndexError`. De array heeft 16000 plaatsen (indices 0 t.e.m. 15999), en je probeert ook plaats 16000 te vullen. Die bestaat niet.

Dit is dé klassieke off-by-one fout. Onthoud: `range(N)` geeft `N` getallen, van `0` tot `N-1`. Niet tot `N`.

### 3

Wat gebeurt er als ik `make_constant_envelope(1.0)` aanroep maar het resultaat *niet* aan een variabele toewijs? 

In [ ]:
make_constant_envelope(1.0)  # geen toewijzing

envelope = make_constant_envelope(1.0)  # toewijzing aan variabele

Zonder het resultaat van de functie toe te wijzen aan een variabele, gaat het verloren. Als je functie *niets* lijkt te doen, vraag je dan altijd af: heb ik het resultaat ergens opgevangen?

### 🟦 Oefening 2.1

Start met deze code:


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd

SAMPLERATE = 16000
duration = 1.0
t = np.linspace(0, duration, int(SAMPLERATE * duration), endpoint=False)
wave = np.sin(2 * np.pi * 220 * t)

def make_silence(duration):
  return ?

all = np.concatenate([wave, make_silence(0.5), wave])

sd.play(all, SAMPLERATE)
sd.wait()

Vervolledig de functie `make_silence(duration)` die een envelope teruggeeft die overal 0 is — dus een stille periode. Daarna plakken we samples aan mekaar met `np.concatenate([wave, make_silence(0.5), wave])`. Dat plakt twee tonen aan elkaar met een halve seconde stilte ertussen.

*Hint:* je kan dit met een `for`-lus, of in één regel met `np.zeros`. Doe het eerst met de lus.

## 3. Lineair uitdoven

Nu iets dat *wel* effect heeft. Bij elke iteratie maken we de waarde een beetje kleiner, lineair van 1 naar 0.

In [ ]:
def make_linear_decay(duration):
    n_samples = int(SAMPLERATE * duration)
    envelope = np.zeros(n_samples)
    for n in range(n_samples):
        envelope[n] = 1.0 - n / n_samples
    return envelope

env = make_linear_decay(1.0)

plt.figure(figsize=(10, 3))
plt.plot(t, env)
plt.xlabel("Tijd (s)")
plt.ylabel("Amplitude")
plt.title("Lineair uitdovende envelope")
plt.show()

Beluister het effect op een sinus van 220 Hz:

In [ ]:
wave = np.sin(2 * np.pi * 220 * t)
sd.play(wave * env, SAMPLERATE)
sd.wait()

In de lijn `envelope[n] = 1.0 - n / n_samples`: bij welke `n` is `envelope[n]` gelijk aan exact 0.5? Reken eerst uit met de hand, controleer dan met code.

In [ ]:
# controle:
n_samples = len(env)
half_index = n_samples // 2
print(f"index {half_index}: envelope = {env[half_index]}")

Bij `n = n_samples / 2`, dus halverwege. Logisch — het is een *lineaire* afname.

Welke tijd in seconden is `t[n]` bij de 1600ste iteratie (`n = 1600`), ? En welke amplitude heeft de envelope dan?

Reken het uit. De samplerate is 16000.

In [ ]:
print(f"t[1600]   = {t[1600]} s")
print(f"env[1600] = {env[1600]}")

`t[1600] = 1600 / 16000 = 0.1` seconden. De envelope op die plek is `1 - 1600/16000 = 0.9`. We zitten dus pas heel vroeg in het uitdoven.

Dit is belangrijk: er is een vast verband tussen sample-index en tijd, namelijk $t = n / \text{SAMPLERATE}$. Of omgekeerd: $n = t \cdot \text{SAMPLERATE}$. Daar maken we vanaf nu vaak gebruik van.

### 🟦 Oefening 3.1 — Omgekeerd


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd

SAMPLERATE = 16000
duration = 1.0
t = np.linspace(0, duration, int(SAMPLERATE * duration), endpoint=False)
wave = np.sin(2 * np.pi * 220 * t)

def make_linear_attack(duration):
  return ?

envelope = make_linear_attack(1.0)

sd.play(wave * envelope, SAMPLERATE)


Vul de functie `make_linear_attack(duration)` aan zodat die het *omgekeerde* doet: een envelope die lineair *opbouwt* van 0 naar 1.

### 🟦 Oefening 3.2 — Niet vanaf 0

Pas `make_linear_attack` aan zodat ze niet bij 1.0 begint en op 0.0 eindigt, maar bij een gekozen `start`-waarde en `end`-waarde. Signatuur: `make_linear_attack(duration, start, end)`. Test met `start=0.3, end=1.0` (zwelt aan vanuit half-luid) en met `start=1.0, end=0.4` (sterft uit, maar niet helemaal).

## 4. Attack + Decay

Een echte instrument-envelope heeft minstens twee fases:

* **Attack**: hoe snel komt het geluid op gang? Een trommelstok raakt het vel meteen (snelle attack). Een strijkstok over een snaar bouwt langzaam op (trage attack).
* **Decay**: hoe sterft het uit? Een gitaarsnaar trilt nog seconden door, een handgeklap is meteen weg.

We maken dit in twee fases:

* van sample `0` tot `attack_samples`: lineair stijgen van 0 naar 1
* van `attack_samples` tot het einde: lineair dalen van 1 naar 0

Lees eerste de code. Hoe gaat deze attack eruit zien?

In [ ]:
def make_ad_envelope(duration, attack, decay):
    n_samples = int(SAMPLERATE * duration)
    attack_samples = int(SAMPLERATE * attack)
    decay_samples = int(SAMPLERATE * decay)
    
    envelope = np.zeros(n_samples)
    for n in range(n_samples):
        if n < attack_samples:
            envelope[n] = n / attack_samples
        elif n < attack_samples + decay_samples:
            envelope[n] = 1.0 - (n - attack_samples) / decay_samples
    return envelope

env = make_ad_envelope(duration=1.0, attack=0.1, decay=0.9)

plt.figure(figsize=(10, 3))
plt.plot(t, env)
plt.xlabel("Tijd (s)")
plt.ylabel("Amplitude")
plt.title("AD-envelope: attack 0.1 s, decay 0.9 s")
plt.show()

Bij `attack = 0.1` en `duration = 1.0`: bij welke sample-index `n` springt de envelope van "stijgend" naar "dalend"? Reken eerst uit (gebruik de samplerate), controleer dan.

In [ ]:
# controle:
attack_samples = int(SAMPLERATE * 0.1)
print(f"omschakelpunt: n = {attack_samples}")
print(f"envelope rond dat punt: {env[attack_samples-1]:.4f}, {env[attack_samples]:.4f}, {env[attack_samples+1]:.4f}")

Plotten met drie verschillende attack-tijden, om het verschil te zien:

In [ ]:
plt.figure(figsize=(10, 6))
for i, attack in enumerate([0.01, 0.1, 0.5]):
    plt.subplot(3, 1, i + 1)
    env = make_ad_envelope(1.0, attack, 1.0 - attack)
    plt.plot(t, env)
    plt.ylabel(f"attack={attack}")
    plt.ylim(-0.05, 1.1)
plt.xlabel("Tijd (s)")
plt.tight_layout()
plt.show()

Start opnieuw vanuit oefening 3. Vervang de functie door `make_ad_envelope(duration, attack, decay)`.

### 🟦 Oefening 4.1 — Pluk vs. strijker

Maak met `make_ad_envelope` twee duidelijk verschillende geluiden van één seconde op 440 Hz:

* attack = 0.05 s, decay = 0.5 s
* attack = 0.3 s, decay = 3.0 s

Speel beide af. Beschrijf in eigen woorden wat je hoort. Welke past bij welk instrument? Wat gebeurt er als je decay langer is dan de duur van de envelope?

### 🟦 Oefening 4.2 — Trommel

Een trommel heeft (1) een vrijwel onmiddellijke attack en (2) een korte totale duur. Pas dat toe op een lage sinus (80 Hz) — een lage frequentie omdat trommels meestal laag zijn.

Klinkt het als een trommel? Wat zou er anders moeten? (Spoiler: een echte trommel heeft een *exponentiële* decay, niet lineair. Daar komen we straks op terug. Bovendien heb je boventonen nodig om het echt te laten klinken.)

### 🟦 Oefening 4.3 — Drumloop

Maak met `np.concatenate` een ritme van vier trommels na mekaar. Kan je dit ook sneller of langzamer laten gaan?

## 5. Dezelfde envelope, maar zonder lus

De lus-versie van `make_ad_envelope` werkt prima. Maar numpy heeft een veel kortere manier: `np.linspace` voor elk stuk, en `np.concatenate` om ze aan elkaar te plakken.

In [ ]:
def make_ad_envelope_numpy(duration, attack, decay):
    n_samples = int(SAMPLERATE * duration)
    attack_samples = int(SAMPLERATE * attack)
    decay_samples = int(SAMPLERATE * decay)
    
    attack_part = np.linspace(0, 1, attack_samples)
    decay_part = np.linspace(1, 0, decay_samples)
    silence_part = np.zeros(n_samples - attack_samples - decay_samples)
    
    return np.concatenate([attack_part, decay_part, silence_part])

env_loop = make_ad_envelope(1.0, 0.1, 0.9)
env_numpy = make_ad_envelope_numpy(1.0, 0.1, 0.9)

plt.figure(figsize=(10, 3))
plt.plot(t, env_loop, label="lus", linewidth=3, alpha=0.5)
plt.plot(t, env_numpy, label="numpy", linewidth=1, color="red")
plt.legend()
plt.xlabel("Tijd (s)")
plt.show()

De twee curves liggen op elkaar — zelfde resultaat, andere manier.

Maar welke versie is sneller? Een kwantitatieve test:

In [ ]:
import time

start = time.time()
for _ in range(100):
    make_ad_envelope(1.0, 0.1, 0.9)
print(f"lus-versie: {time.time() - start:.3f} s")

start = time.time()
for _ in range(100):
    make_ad_envelope_numpy(1.0, 0.1, 0.9)
print(f"numpy-versie: {time.time() - start:.3f} s")

Het verschil is fors — typisch 30 tot 100 keer sneller. Voor één envelope merk je het niet, maar als je honderden noten genereert (of langere stukken audio) wel. Daarom: voor productie numpy, maar de lus blijft nuttig om te begrijpen *wat* er gebeurt.

### 🟦 Oefening 5.1 — Exponentiële decay

In les 1 hebben we exponentiële demping gezien: `np.exp(-decay_rate * t)` daalt sneller in het begin en zachter naar het einde. Dat is realistischer.

Schrijf `make_exp_envelope(duration, attack, decay, decay_rate=5.0)` waarbij:

* de attack lineair stijgt (zoals voorheen)
* de decay een exponentiële vorm heeft

*Hint:* `t` staat in de formule voor tijd. Je zal dus opnieuw linspace moeten gebruiken.

Test met `decay_rate = 5.0` en vergelijk met de lineaire versie. Welke klinkt natuurlijker?

Je kan starten met deze code:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sounddevice as sd

SAMPLERATE = 16000
duration = 1.0
t = np.linspace(0, duration, int(SAMPLERATE * duration), endpoint=False)
wave = np.sin(2 * np.pi * 220 * t)

def make_exp_envelope(duration, attack, decay, decay_rate=5.0):
  n_samples = int(SAMPLERATE * duration)
  attack_samples = int(SAMPLERATE * attack)
  decay_samples = int(SAMPLERATE * decay)
    
  attack_part = np.linspace(0, 1, attack_samples)
  decay_part = # TODO: maak een exponentieel verval van 1 naar 0 over decay_samples, met decay_rate als verval-snelheid
  silence_part = np.zeros(n_samples - attack_samples - decay_samples)
    
  return np.concatenate([attack_part, decay_part, silence_part])

envelope = make_exp_envelope(1.0, 0.1, 0.9)

sd.play(wave * envelope, SAMPLERATE)

## 6. Een mini-instrumentarium

Tijd om alles samen te brengen. We koppelen onze envelope aan een geluidsbron (de sinus, of de harmonischen-aanpak van les 2) en bouwen verschillende karakters.

In [ ]:
def make_sine(frequency, duration):
    t = np.linspace(0, duration, int(SAMPLERATE * duration), endpoint=False)
    return np.sin(2 * np.pi * frequency * t)

def my_drum(frequency=120, duration=1.0):
    wave = make_sine(frequency, duration)
    envelope = make_ad_envelope_numpy(duration, 0.001, 0.2)
    return wave * envelope

def my_pluck(frequency=220, duration=1.0):
    wave = make_sine(frequency, duration)
    envelope = make_ad_envelope_numpy(duration, 0.005, 0.8)
    return wave * envelope

def my_bow(frequency=440, duration=1.0):
    wave = make_sine(frequency, duration)
    envelope = make_ad_envelope_numpy(duration, 0.4, 0.6)
    return wave * envelope

In [ ]:
# beluister achter elkaar:
sequence = np.concatenate([
    my_drum(),
    np.zeros(int(SAMPLERATE * 0.3)),
    my_pluck(),
    np.zeros(int(SAMPLERATE * 0.3)),
    my_bow(),
])
sd.play(sequence, SAMPLERATE)
sd.wait()

Drie keer dezelfde golfvorm, drie totaal verschillende karakters — enkel door de envelope.

### 🟦 Oefening 6.1 — Eigen instrument

Combineer de **harmonischen-aanpak** uit les 2 (`my_instrument`) met een AD-envelope. Je kan de functie my_instrument uit de vorige les hier terug toevoegen. Schrijf dan een functie `my_xylophone(frequency, duration=0.3)` die:

* je instrument gebruikt (kies zelf de amplitudes — een xylofoon heeft sterke hogere harmonischen)
* een enveloppe maakt (een snelle attack (≈ 0.002 s) en korte duur)

En dan enkele tonen speelt.

### 🟦 Oefening 6.2 — Een ritmisch motief

Maak een eenvoudig ritme: vier noten van 0.25 s elk, met je eigen instrument. Maak de eerste en derde luider dan de tweede en vierde door het signaal met een factor te vermenigvuldigen (bv. 1.0 vs. 0.5).


### 🟦 Oefening 6.3 (bonus) — ASR-envelope

Voeg een **sustain**-fase toe tussen attack en decay. Bij sustain blijft de amplitude een tijdje constant op een gekozen niveau (bv. 0.7), pas daarna komt de decay. Schrijf:

```python
def make_asr_envelope(duration, attack, sustain, decay, level=0.7):
    ...
```

Waarbij `duration`, `attack`, `sustain` en `decay` elk een duur in seconden zijn, en `level` het sustain-niveau (tussen 0 en 1).

Test op een fluit-achtig geluid (langere sustain, weinig harmonischen).